# Develop a Multi-Agent Research Assistant Using AutoGen

### Task 0: Set up Your Environment and Load API Keys

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")


In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key=api_key
)



### Task 1: Import Libraries

In [3]:
from typing import List , Sequence
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
import requests
import xml.etree.ElementTree as ET


### Task 2: Implement an arXiv Paper Search Function

In [4]:
def arxiv_Search_tool(query: str, max_results:int =5) -> str:
    base_url = "http://export.arxiv.org/api/query"

    params = {
        "search_query": query,
        "start": 0,
        "max_results": max_results
    }

    try:
        response = requests.get(base_url, params=params)

        if response.status_code != 200:
            return f"Error: Failed to fetch data (status code {response.status_code})"

        root = ET.fromstring(response.content)

        # Parse XML response
        root = ET.fromstring(response.content)

        entries = root.findall("{http://www.w3.org/2005/Atom}entry")

        if not entries:
            return "No papers found for this query."

        results = []

        for entry in entries:
            title = entry.find("{http://www.w3.org/2005/Atom}title").text.strip()
            # {XML namespace}tagname
            '''
            <entry xmlns="http://www.w3.org/2005/Atom">
    <title>Some Paper Title</title>
</entry>
            '''
            summary = entry.find("{http://www.w3.org/2005/Atom}summary").text.strip()
            link = entry.find("{http://www.w3.org/2005/Atom}id").text.strip()

            paper_info = f"""
Title: {title}
Link: {link}
Abstract: {summary[:300]}...
"""
            results.append(paper_info)

        return "\n\n".join(results)

    except Exception as e:
        return f"Error occurred: {str(e)}"

### Task 3: Create the Topic Refinement Agent

In [5]:
topic_agent = AssistantAgent(
    name="topic_agent",
    model_client=model_client,
    description="An AI agent that refines broad research ideas into specific research topics.",
    
    system_message="""
You are a research topic refinement assistant.

Your job is to take a broad or vague research idea and refine it into exactly ONE clear, specific, and focused research topic.

Rules:
- Only output ONE refined topic
- Be precise and academically relevant
- Do not provide explanations

Output format:
Refined Research Topic: <your topic>

End your response with:
TASK_COMPLETED_TOPIC_AGENT
"""
)

In [6]:
response = await topic_agent.run(
    task="I want to research AI in healthcare"
)

print(response)

messages=[TextMessage(id='eea24cf9-679b-44d4-b7c9-0a73ff287819', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 2, 41, 113706, tzinfo=datetime.timezone.utc), content='I want to research AI in healthcare', type='TextMessage'), TextMessage(id='d98437f7-1224-463a-a02f-f4847d0b14a2', source='topic_agent', models_usage=RequestUsage(prompt_tokens=99, completion_tokens=24), metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 2, 42, 851402, tzinfo=datetime.timezone.utc), content='Refined Research Topic: The impact of AI-driven predictive analytics on diagnostic accuracy in cardiology.\n\nTASK_COMPLETED_TOPIC_AGENT', type='TextMessage')] stop_reason=None


### Task 4: Create the Paper Discovery Agent

In [7]:
paper_agent = AssistantAgent(
    name="paper_agent",
    model_client=model_client,
    tools=[arxiv_Search_tool],  # ✅ connect your function
    
    description="An AI agent that searches and retrieves relevant research papers from arXiv.",
    
    system_message="""
You are a research paper discovery assistant.

Your task is to:
- Take a research topic as input
- Use the arxiv_search_tool to find relevant academic papers
- Return a structured summary of the results

Rules:
- Always use the arxiv_search_tool to fetch papers
- Return multiple relevant papers
- Keep summaries concise and readable

Output format:
Paper 1:
Title: <title>
Link: <url>
Summary: <short summary>

Paper 2:
...

End your response with:
TASK_COMPLETED_PAPER_AGENT
"""
)

In [8]:
response = await paper_agent.run(
    task="AI in healthcare"
)

print(response)

messages=[TextMessage(id='d218eb5c-f67b-415e-86e3-6b8a0a8e07a6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 2, 43, 460473, tzinfo=datetime.timezone.utc), content='AI in healthcare', type='TextMessage'), ToolCallRequestEvent(id='687183b0-5f4e-40aa-bc45-715e38def9c2', source='paper_agent', models_usage=RequestUsage(prompt_tokens=178, completion_tokens=23), metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 2, 48, 368929, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_1QcUOBVT8xhini6K6qUm9Q6w', arguments='{"query":"AI in healthcare","max_results":5}', name='arxiv_Search_tool')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='d7c6828d-d6f5-4179-97c0-e461ef6e4680', source='paper_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 2, 49, 172584, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content="\nTitle: An Overview and Case Study of the Clinical AI Model De

### Task 5: Add the Insight Synthesizer Agent

In [9]:
insight_agent = AssistantAgent(
    name="insight_agent",
    model_client=model_client,
    
    description="An AI agent that analyzes research papers and extracts key insights and trends.",
    
    system_message="""
You are a research insight synthesizer.

Your task is to:
- Read research paper summaries provided as input
- Identify key themes, trends, and findings across the papers
- Summarize them into clear insights

Rules:
- Provide 3 to 5 concise bullet points
- Focus on common patterns or important findings
- Do not repeat paper titles
- Keep insights high-level and meaningful

Output format:
- Insight 1
- Insight 2
- Insight 3
...

End your response with:
TASK_COMPLETED_INSIGHT_AGENT
"""
)

In [11]:
paper_results = await paper_agent.run(
    task="AI in healthcare"
)

insights = await insight_agent.run(
    task=str(paper_results)
)

print(insights.messages[-1].content)

- AI model development in healthcare requires stakeholder engagement and education to ensure successful adoption and alignment with healthcare needs, highlighting the importance of a multidisciplinary approach.
- The integration of AI in decision-making processes can significantly influence behavior, as evidenced by studies showing AI's impact on decision paradigms, which underscores the need to understand AI's role in shaping human decisions.
- Advanced generative AI models contribute notably to the improvement of information systems by delivering more human-like and contextually appropriate responses, marking a shift from traditional data-driven AI to more sophisticated interactions.
- Security remains a paramount concern with autonomous AI agents in healthcare, necessitating robust architectures like zero-trust models to safeguard sensitive operations and data from vulnerabilities inherent in complex AI systems.
- The discourse on ethical AI adoption varies among organizations, emph

### Task 6: Create the Report Compiler Agent

In [12]:
report_agent = AssistantAgent(
    name="report_agent",
    model_client=model_client,
    
    description="An AI agent that compiles research insights into a structured and readable report.",
    
    system_message="""
You are a research report compiler.

Your task is to:
- Take summarized research insights as input
- Organize them into a well-structured report

The report must include the following sections:

1. Introduction
- Brief overview of the topic

2. Key Findings
- Present the insights clearly (can use bullet points)

3. Discussion
- Explain implications, patterns, or significance

4. Conclusion
- Summarize the overall understanding

Rules:
- Keep the report clear and concise
- Maintain a formal tone
- Do not add unrelated information

End your response with:
TASK_COMPLETED_REPORT_AGENT
"""
)

In [13]:
insight_output = insights.messages[-1].content

report = await report_agent.run(
    task=insight_output
)

print(report.messages[-1].content)

**Research Report: The Development and Integration of AI in Modern Systems**

**1. Introduction**

Artificial Intelligence (AI) is becoming an integral part of various sectors, with its applications in healthcare gaining particular attention. This report explores the development of AI models, their integration into decision-making processes, and the critical factors for their adoption, including stakeholder engagement, security, and ethical considerations.

**2. Key Findings**

- Successful AI model development in healthcare requires the engagement and education of stakeholders, demonstrating a need for multidisciplinary collaboration.
- The integration of AI into decision-making significantly influences human behavior, underscoring the necessity to comprehend the dynamics of AI-human interactions.
- Generative AI models enhance information systems by providing human-like and contextually accurate responses, indicating a transition from traditional methodologies to more sophisticated A

### Task 7: Add the Gap Analysis Agent

In [14]:
gap_agent = AssistantAgent(
    name="gap_agent",
    model_client=model_client,
    
    description="An AI agent that analyzes research reports to identify gaps and suggest future research directions.",
    
    system_message="""
You are a research gap analysis expert.

Your task is to:
- Analyze a given research report
- Identify missing areas, limitations, or unexplored aspects
- Suggest meaningful future research directions

Output format:

Research Gaps:
- Gap 1
- Gap 2
- Gap 3

Future Directions:
- Direction 1
- Direction 2
- Direction 3

Rules:
- Be specific and insightful
- Avoid repeating existing findings
- Focus on what is missing or needs improvement

End your response with:
FINAL REPORT
"""
)

In [15]:
report_output = report.messages[-1].content

gap_analysis = await gap_agent.run(
    task=report_output
)

print(gap_analysis.messages[-1].content)

Research Gaps:
- Gap 1: Limited exploration of the impact of AI on specific professional roles within healthcare, particularly how AI models might alter existing job functions, create new roles, or lead to workforce displacement.
- Gap 2: Insufficient focus on the long-term sustainability and performance of AI systems in healthcare environments, particularly how these systems adapt and maintain accuracy over time amidst evolving datasets and clinical practices.
- Gap 3: Lack of a comprehensive framework for measuring the effectiveness and success of AI integration across different sectors beyond healthcare, which could provide insights into cross-industry AI adoption strategies and outcomes.

Future Directions:
- Direction 1: Conduct detailed studies on the socio-economic impacts of AI on healthcare professionals, examining changes in workforce dynamics, job satisfaction, and skill requirements as AI becomes more integrated.
- Direction 2: Investigate the longitudinal effects of AI imp

### Task 8: Define Termination Conditions

In [16]:
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
text_termination = TextMentionTermination("FINAL REPORT")
max_messages_termination = MaxMessageTermination(max_messages=10)
termination_condition = text_termination | max_messages_termination

### Task 9: Build the Multi-Agent Workflow

In [18]:
team = RoundRobinGroupChat(
    participants=[
        topic_agent,
        paper_agent,
        insight_agent,
        report_agent,
        gap_agent
    ],
    termination_condition=termination_condition
)
task = "AI for healthcare"

await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
AI for healthcare
---------- TextMessage (topic_agent) ----------
Refined Research Topic: Evaluating the ethical implications of using AI algorithms in patient data management within digital health records.

TASK_COMPLETED_TOPIC_AGENT
---------- ToolCallRequestEvent (paper_agent) ----------
[FunctionCall(id='call_HbAfgAlpSwq1mFwDinUDj9Js', arguments='{"query":"AI for healthcare","max_results":5}', name='arxiv_Search_tool')]
---------- ToolCallExecutionEvent (paper_agent) ----------
[FunctionExecutionResult(content="\nTitle: An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems\nLink: http://arxiv.org/abs/2003.07678v3\nAbstract: Healthcare is one of the most promising areas for machine learning models to make a positive impact. However, successful adoption of AI-based systems in healthcare depends on engaging and educating stakeholders from diverse backgrounds about the development process of AI models.

TaskResult(messages=[TextMessage(id='7e93ee75-600e-4d10-bcde-15ef14d39059', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 16, 4, 359452, tzinfo=datetime.timezone.utc), content='AI for healthcare', type='TextMessage'), TextMessage(id='ef4a3553-6240-4cea-9c25-d46cbbf90925', source='topic_agent', models_usage=RequestUsage(prompt_tokens=135, completion_tokens=28), metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 16, 6, 385431, tzinfo=datetime.timezone.utc), content='Refined Research Topic: Evaluating the ethical implications of using AI algorithms in patient data management within digital health records.\n\nTASK_COMPLETED_TOPIC_AGENT', type='TextMessage'), ToolCallRequestEvent(id='3bcd71ff-c92e-46f2-8236-598ec8910a02', source='paper_agent', models_usage=RequestUsage(prompt_tokens=1419, completion_tokens=23), metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 16, 7, 845346, tzinfo=datetime.timezone.utc), content=[FunctionCall(id=

### Task 10: Insert a User Proxy Agent for Manual Approval

In [19]:
user_proxy_agent = UserProxyAgent(
    name="user_proxy_agent",
    description="A human-in-the-loop agent that reviews and approves outputs before proceeding to the next step."
)

### Task 11: Implement a Custom Selector Function for User Approval

In [26]:
user_approved = False
def selector_func_with_user_proxy(messages):
    global user_approved

    last_message = messages[-1].content if messages else ""
    last_source = messages[-1].source if messages else ""

    # Step 1: Start
    if len(messages) == 0:
        return "topic_agent"

    # Step 2: After topic_agent → user approval
    if last_source == "topic_agent" and not user_approved:
        return "user_proxy_agent"

    # Step 3: Handle approval
    if last_source == "user_proxy_agent":
        user_input = last_message.upper()

        if "APPROVE" in user_input:
            user_approved = True
            return "paper_agent"

        elif "DISAPPROVE" in user_input:
            user_approved = False
            return "topic_agent"

    # Step 4: Continue flow
    if last_source == "paper_agent":
        return "insight_agent"

    if last_source == "insight_agent":
        return "report_agent"

    if last_source == "report_agent":
        return "gap_agent"

    return None

### Task 12: Execute the Full Interactive Research Workflow

In [28]:
selector_prompt = """
You are coordinating a multi-agent research workflow.

Agents involved:
- topic_agent: Refines the research topic
- user_proxy_agent: Gets user approval (APPROVE or DISAPPROVE)
- paper_agent: Finds research papers
- insight_agent: Extracts insights
- report_agent: Creates report
- gap_agent: Performs gap analysis

Rules:
- Start with topic_agent
- After topic_agent, ask user_proxy_agent for approval
- If user says APPROVE → continue workflow
- If user says DISAPPROVE → go back to topic_agent
- Continue agents in order after approval
- Stop when FINAL REPORT is produced
"""

In [29]:
task = "AI for healthcare"

In [30]:
team = SelectorGroupChat(
    participants=[
        topic_agent,
        user_proxy_agent,
        paper_agent,
        insight_agent,
        report_agent,
        gap_agent
    ],
    model_client=model_client,
    termination_condition=termination_condition,
    selector_prompt=selector_prompt,
    selector_func=selector_func_with_user_proxy,
    allow_repeated_speaker=False
)

In [31]:
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
AI for healthcare
---------- TextMessage (user_proxy_agent) ----------
APPROVE
---------- TextMessage (paper_agent) ----------
Here are some relevant academic papers on the topic of AI for healthcare:

Paper 1:
Title: An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems
Link: [http://arxiv.org/abs/2003.07678v3](http://arxiv.org/abs/2003.07678v3)
Summary: This paper presents a case study of AI model development within healthcare systems, highlighting the importance of engaging diverse stakeholders in the process to ensure effective implementation and adoption of AI technologies in healthcare.

Paper 2:
Title: Caging the Agents: A Zero Trust Security Architecture for Autonomous AI in Healthcare
Link: [http://arxiv.org/abs/2603.17419v1](http://arxiv.org/abs/2603.17419v1)
Summary: Discusses the deployment of autonomous AI agents using large language models in healthcare. It addresses the vulnerabilities o

TaskResult(messages=[TextMessage(id='7b191fc4-e8fe-430e-903e-2b575d2aba15', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 33, 41, 376294, tzinfo=datetime.timezone.utc), content='AI for healthcare', type='TextMessage'), UserInputRequestedEvent(id='28555444-2302-4143-b89c-9fff27be94bb', source='user_proxy_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 33, 44, 161333, tzinfo=datetime.timezone.utc), request_id='c6566da2-bf92-4f61-874d-1381ad311997', content='', type='UserInputRequestedEvent'), TextMessage(id='f8a65e66-3f73-43b8-82f4-ea4500a61f6e', source='user_proxy_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 8, 17, 33, 49, 480571, tzinfo=datetime.timezone.utc), content='APPROVE', type='TextMessage'), TextMessage(id='a13f5567-5c68-4d6c-a9a9-7a47e946390e', source='paper_agent', models_usage=RequestUsage(prompt_tokens=1910, completion_tokens=228), metadata={}, created_at=date